# RecursiveMAAT v9.3 -- TOFU forget05/retain95

**Dataset**: `locuslab/TOFU` -- `forget05` (200 samples) / `retain95` (3800 samples)  
**Label setup**: None (TOFU is plain Q&A -- no ClassLabel, no stratified sampling)  
**Platform**: Kaggle T4x2  
**Adapter**: `Novaspree/llama-3.2-3B-tofu-adapter`

### Changes & fixes vs factify-v9.3-500

| # | Where | Change / Bug | Fix |
|---|---|---|---|
| 1 | Section 1 | factify 5-label dataset | Replaced with `locuslab/TOFU` forget05/retain95 |
| 2 | Section 2 | `Novaspree/factify-3B-adapter` | Changed to `Novaspree/llama-3.2-3B-tofu-adapter` |
| 3 | **New** | No pre-unlearning baseline | Added **BERTScore on forget05** before unlearning |
| 4 | Engine Phase 3 | `tv_sims` never aggregated -> TV reg was always 0 | `tv_reg_loss = torch.stack(tv_sims).mean()` |
| 5 | Engine Phase 3 | No `clip_grad_norm_` before `opt.step()` | Added |
| 6 | Section 6 | `label_feature.int2str(sample['label'])` crashes on TOFU | Removed label field |

## Setup

In [ ]:
!pip install -q trl bitsandbytes accelerate peft datasets transformers huggingface_hub evaluate rouge_score bert_score

## Authentication

In [ ]:
import os, torch
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN')
login(hf_token)
print(f'GPUs: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB')

## 1 . Dataset Preparation

Loads `forget05` (200 samples) and `retain95` (3800 samples) from `locuslab/TOFU`.
TOFU has no label column -- no ClassLabel, no stratified sub-sampling.
Each sample is wrapped with the Llama-3 chat template and saved to disk.

In [ ]:
import os
from datasets import load_dataset

DATA_DIR = '/kaggle/working/data_splits'
os.makedirs(DATA_DIR, exist_ok=True)

print('[1] Loading TOFU forget05 / retain95 splits...')
forget_dataset = load_dataset('locuslab/TOFU', 'forget05', split='train')
retain_dataset = load_dataset('locuslab/TOFU', 'retain95', split='train')

print(f'Forget (forget05): {len(forget_dataset)} samples')
print(f'Retain (retain95): {len(retain_dataset)} samples')
print(f'Features         : {list(forget_dataset.features.keys())}')

def format_for_training(example):
    return {'text': (
        '<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n'
        f"{example['question']}<|eot_id|>"
        '<|start_header_id|>assistant<|end_header_id|>\n\n'
        f"{example['answer']}<|eot_id|>"
    )}

forget_dataset = forget_dataset.map(format_for_training)
retain_dataset = retain_dataset.map(format_for_training)

forget_dataset.save_to_disk(f'{DATA_DIR}/forget_dataset')
retain_dataset.save_to_disk(f'{DATA_DIR}/retain_dataset')
print(f'\nSaved  forget={len(forget_dataset)}  retain={len(retain_dataset)}')
print(f'Features: {list(forget_dataset.features.keys())}')

## 2 . Load Pre-trained Adapter

`Novaspree/llama-3.2-3B-tofu-adapter` was fine-tuned on the full TOFU 4K dataset:
- Base: `meta-llama/Llama-3.2-3B`, 4-bit NF4 quantisation
- LoRA: rank=32, alpha=64, all 7 modules, layers 7-20, 3 epochs

Unlearning loads the adapter in **fp16** (T4-safe).

In [ ]:
import gc, shutil, os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from huggingface_hub import snapshot_download

MODEL_NAME        = 'meta-llama/Llama-3.2-3B'
ADAPTER_REPO      = 'Novaspree/llama-3.2-3B-tofu-adapter'
LORA_ADAPTER_PATH = '/kaggle/working/lora_adapter'

gc.collect()
torch.cuda.empty_cache()

print(f'[2] Downloading adapter from {ADAPTER_REPO}...')
if os.path.exists(LORA_ADAPTER_PATH):
    shutil.rmtree(LORA_ADAPTER_PATH)
snapshot_download(repo_id=ADAPTER_REPO, local_dir=LORA_ADAPTER_PATH)
print(f'Adapter saved -> {LORA_ADAPTER_PATH}')

try:
    tokenizer = AutoTokenizer.from_pretrained(LORA_ADAPTER_PATH)
    print('Tokenizer loaded from adapter repo')
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    print('Tokenizer loaded from base model')
    tokenizer.save_pretrained(LORA_ADAPTER_PATH)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.save_pretrained(LORA_ADAPTER_PATH)

print('\nSanity check -- loading PeftModel...')
_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map='auto'
)
_peft = PeftModel.from_pretrained(_base, LORA_ADAPTER_PATH, is_trainable=False)
n_trainable = sum(p.numel() for p in _peft.parameters() if p.requires_grad)
print(f'Trainable params (adapter only): {n_trainable:,}')
del _peft, _base
gc.collect(); torch.cuda.empty_cache()
print('Sanity check passed.')

## 3 . BERTScore Baseline on forget05 (PRE-unlearning)

Generates model answers on all 200 `forget05` questions using the **fine-tuned adapter**
and computes BERTScore (Precision / Recall / F1) against the ground-truth answers.

This establishes the baseline: how well does the adapter currently know the forget set?
After unlearning, we expect BERTScore on forget05 to **drop** while retain95 stays high.

In [ ]:
import json, gc, os
import torch
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from bert_score import score as bert_score_fn

MODEL_NAME        = 'meta-llama/Llama-3.2-3B'
LORA_ADAPTER_PATH = '/kaggle/working/lora_adapter'
DATA_DIR          = '/kaggle/working/data_splits'
OUT_DIR           = '/kaggle/working/eval_inputs'
os.makedirs(OUT_DIR, exist_ok=True)

gc.collect(); torch.cuda.empty_cache()

# Load the fine-tuned (pre-unlearn) model for baseline generation
print('[3] Loading fine-tuned adapter for BERTScore baseline...')
tokenizer_bs = AutoTokenizer.from_pretrained(LORA_ADAPTER_PATH)
if tokenizer_bs.pad_token is None:
    tokenizer_bs.pad_token = tokenizer_bs.eos_token

base_bs = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map='auto'
)
model_bs = PeftModel.from_pretrained(base_bs, LORA_ADAPTER_PATH, is_trainable=False)
model_bs.eval()
bs_device = next(model_bs.parameters()).device

eos_ids_bs = list({tokenizer_bs.eos_token_id,
                   tokenizer_bs.convert_tokens_to_ids('<|eot_id|>')})
eos_ids_bs = [e for e in eos_ids_bs if e is not None and e >= 0]


def _bs_prompt(question):
    return (
        '<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n'
        f'{question}<|eot_id|>'
        '<|start_header_id|>assistant<|end_header_id|>\n\n'
    )


def _bs_generate(question):
    import re
    prompt = _bs_prompt(question)
    inp = tokenizer_bs(prompt, return_tensors='pt',
                       truncation=True, max_length=512).to(bs_device)
    with torch.no_grad():
        gen = model_bs.generate(
            **inp,
            max_new_tokens=100, do_sample=False,
            repetition_penalty=1.3,
            eos_token_id=eos_ids_bs,
            pad_token_id=tokenizer_bs.eos_token_id,
        )
    raw = tokenizer_bs.decode(
        gen[0][inp['input_ids'].shape[1]:], skip_special_tokens=True
    ).strip()
    return re.sub(r'\s*(CLIIIK|\?>|\];|\]|/\*|<!|\{\{|\}\}).*$',
                  '', raw, flags=re.DOTALL).strip()


forget_ds_bs = load_from_disk(f'{DATA_DIR}/forget_dataset')
print(f'Generating answers for {len(forget_ds_bs)} forget05 samples...')

preds_bs, refs_bs, questions_bs = [], [], []
for i, sample in enumerate(forget_ds_bs):
    pred = _bs_generate(sample['question'])
    preds_bs.append(pred)
    refs_bs.append(sample['answer'])
    questions_bs.append(sample['question'])
    if (i + 1) % 20 == 0:
        torch.cuda.empty_cache()
        print(f'  [{i+1}/{len(forget_ds_bs)}]')

# Free the baseline model before unlearning
del model_bs, base_bs
gc.collect(); torch.cuda.empty_cache()
print('Baseline model freed.')

# Compute BERTScore (roberta-large by default, lang='en')
print('\nComputing BERTScore (this may take 1-2 min)...')
P, R, F1 = bert_score_fn(
    preds_bs, refs_bs,
    lang='en',
    model_type='roberta-large',
    device=str(bs_device),
    verbose=False,
)

bertscore_pre = {
    'stage'     : 'pre_unlearning',
    'split'     : 'forget05',
    'n_samples' : len(forget_ds_bs),
    'precision' : round(P.mean().item(), 4),
    'recall'    : round(R.mean().item(), 4),
    'f1'        : round(F1.mean().item(), 4),
}

print('\n===== BERTScore Baseline (PRE-unlearning, forget05) =====')
print(f'  Precision : {bertscore_pre["precision"]:.4f}')
print(f'  Recall    : {bertscore_pre["recall"]:.4f}')
print(f'  F1        : {bertscore_pre["f1"]:.4f}')
print('  (Higher means the adapter currently knows the forget content well)')

# Save per-sample records for later comparison
bertscore_pre_records = []
for i, (q, pred, ref, p, r, f) in enumerate(
        zip(questions_bs, preds_bs, refs_bs, P.tolist(), R.tolist(), F1.tolist())):
    bertscore_pre_records.append({
        'idx': i, 'question': q,
        'ground_truth': ref, 'model_answer': pred,
        'precision': round(p, 4), 'recall': round(r, 4), 'f1': round(f, 4),
    })

with open(f'{OUT_DIR}/bertscore_pre_unlearning.json', 'w') as fp:
    json.dump({'summary': bertscore_pre, 'records': bertscore_pre_records},
              fp, indent=2, ensure_ascii=False)
print(f'Saved -> {OUT_DIR}/bertscore_pre_unlearning.json')

gc.collect(); torch.cuda.empty_cache()

## 4 . RecursiveMAAT v9.3 Engine

**Bug-fixes vs factify-500:**

1. **TV reg bug** -- `tv_sims` list was built but never aggregated into `tv_reg_loss`;
   TV regulariser was silently 0.0 every step. Fixed with `torch.stack(tv_sims).mean()`.
2. **Missing grad clip** -- `retain_repair` had no `clip_grad_norm_` before `opt.step()`.
   Added.

In [ ]:
import math, re, gc, random
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm


# ---------------------------------------------------------------------------
# Utilities
# ---------------------------------------------------------------------------

def get_input_device(model):
    return next(model.parameters()).device


def format_prompt_eval(question):
    return (
        '<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n'
        f'{question}<|eot_id|>'
        '<|start_header_id|>assistant<|end_header_id|>\n\n'
    )


def make_answer_only_labels(input_ids, prompt_len):
    labels = input_ids.clone()
    labels[:, :prompt_len] = -100
    return labels


_GARBAGE_TOKENS = [
    'PostalCodes', 'URLException', 'ityEngine', 'isContained',
    'endcode', 'UsageId', 'ModelError', 'WebAPI', 'togroup',
    'gamber', 'inertia', 'BranchURL', 'invokeSupportedCont',
]

def is_output_broken(text, severe_only=False):
    if not text or len(text.strip()) < 5:
        return True
    alnum = sum(c.isalnum() or c in ' .,!?-' for c in text)
    if alnum / max(1, len(text)) < 0.35:
        return True
    if severe_only:
        return False
    for tok in _GARBAGE_TOKENS:
        if tok in text:
            return True
    if text.count('?') > 3 and len(text) < 300:
        return True
    words = text.split()
    if words and sum(len(w) for w in words) / len(words) > 14:
        return True
    return False


def clean_output(text):
    text = re.sub(r'\s*(CLIIIK|\?>|\];|\]|/\*|<!|\{\{|\}\}).*$',
                  '', text, flags=re.DOTALL)
    return text.strip()


# ---------------------------------------------------------------------------
# Engine
# ---------------------------------------------------------------------------

class RecursiveMAAT_v9:
    # Bug-fix 1: tv_sims now aggregated into tv_reg_loss
    # Bug-fix 2: clip_grad_norm_ added in retain_repair

    UNLEARN_MODULE_TYPES = ('down_proj', 'up_proj', 'q_proj', 'v_proj')
    REPAIR_MODULE_TYPES  = ('down_proj', 'up_proj', 'gate_proj',
                            'q_proj', 'k_proj', 'v_proj', 'o_proj')
    SVD_MLP_TYPES        = ('down_proj', 'up_proj', 'gate_proj')
    SVD_ATTN_TYPES       = ('o_proj',)
    HIDDEN_LAYER_IDX     = 20

    def __init__(
        self, model,
        mid_layer_start=7, mid_layer_end=20,
        learning_rate=1e-5, max_grad_norm=1.0,
        kl_temp=0.7,
        do_svd_prune=True, svd_prune_ratio=0.15,
        attn_prune_ratio=0.02, attn_prune_layer_start=14,
    ):
        self.model               = model
        self.lr                  = learning_rate
        self.max_grad_norm       = max_grad_norm
        self.kl_temp             = kl_temp
        self.do_svd_prune        = do_svd_prune
        self.svd_prune_ratio     = svd_prune_ratio
        self.attn_prune_ratio    = attn_prune_ratio
        self.attn_prune_ls       = attn_prune_layer_start
        self.mid_layer_start     = mid_layer_start
        self.mid_layer_end       = mid_layer_end
        self.forget_task_vec     = {}

        self.lora_a_modules, self.lora_b_modules = [], []
        for name, mod in model.named_modules():
            if not isinstance(mod, nn.Linear): continue
            m = re.search(r'layers\.(\d+)', name)
            if not m: continue
            if not (mid_layer_start <= int(m.group(1)) <= mid_layer_end): continue
            if not any(mt in name for mt in self.UNLEARN_MODULE_TYPES): continue
            if   'lora_A' in name: self.lora_a_modules.append((name, mod))
            elif 'lora_B' in name: self.lora_b_modules.append((name, mod))
        self.target_modules = self.lora_a_modules + self.lora_b_modules
        if not self.target_modules:
            raise RuntimeError('No lora_A/lora_B modules found. Pass a trainable PeftModel.')

        self.attn_a_modules, self.attn_b_modules = [], []
        for name, mod in model.named_modules():
            if not isinstance(mod, nn.Linear): continue
            m = re.search(r'layers\.(\d+)', name)
            if not m: continue
            layer = int(m.group(1))
            if not (attn_prune_layer_start <= layer <= mid_layer_end): continue
            if not any(mt in name for mt in self.SVD_ATTN_TYPES): continue
            if   'lora_A' in name: self.attn_a_modules.append((name, mod))
            elif 'lora_B' in name: self.attn_b_modules.append((name, mod))

        self.all_lora_mods = [
            (n, mod) for n, mod in model.named_modules()
            if isinstance(mod, nn.Linear) and ('lora_A' in n or 'lora_B' in n)
        ]
        self.finetuned_weights = {
            n: mod.weight.data.clone().cpu() for n, mod in self.all_lora_mods
        }

        for p in model.parameters(): p.requires_grad = False
        for _, mod in self.target_modules: mod.weight.requires_grad = True

        n_unlearn = sum(m.weight.numel() for _, m in self.target_modules)
        n_total   = sum(p.numel() for p in model.parameters())
        print('RecursiveMAAT v9.3 (TOFU edition) ready')
        print(f'  Phase 1 targets  : {len(self.target_modules)} modules '
              f'({len(self.lora_a_modules)}A + {len(self.lora_b_modules)}B)')
        print(f'  Phase 2b o_proj  : {len(self.attn_a_modules)}A + '
              f'{len(self.attn_b_modules)}B  (layers {attn_prune_layer_start}-{mid_layer_end})')
        print(f'  Trainable params : {n_unlearn:,} / {n_total:,} ({100*n_unlearn/n_total:.3f}%)')
        print(f'  lr={learning_rate} | kl_temp={kl_temp} | '
              f'mlp_svd={svd_prune_ratio:.0%} | attn_prune={attn_prune_ratio:.0%}')

    def _encode(self, question, answer, tokenizer, device):
        prompt = format_prompt_eval(question)
        plen   = tokenizer(prompt, return_tensors='pt').input_ids.shape[1]
        enc    = tokenizer(prompt + answer, return_tensors='pt',
                           truncation=True, max_length=512).to(device)
        return enc, make_answer_only_labels(enc['input_ids'], plen)

    def _get_finetuned_logits(self, enc):
        self.model.eval()
        saved = {n: mod.weight.data.clone() for n, mod in self.target_modules}
        for n, mod in self.target_modules:
            w = self.finetuned_weights.get(n)
            if w is not None: mod.weight.data.copy_(w.to(mod.weight.device))
        with torch.no_grad():
            logits = self.model(**enc).logits.float().detach()
        for n, mod in self.target_modules: mod.weight.data.copy_(saved[n])
        del saved
        return logits

    def _get_finetuned_outputs(self, enc):
        self.model.eval()
        saved = {n: mod.weight.data.clone() for n, mod in self.all_lora_mods}
        for n, mod in self.all_lora_mods:
            w = self.finetuned_weights.get(n)
            if w is not None: mod.weight.data.copy_(w.to(mod.weight.device))
        with torch.no_grad():
            out    = self.model(**enc, output_hidden_states=True)
            logits = out.logits.float().detach()
            hs_ref = out.hidden_states[self.HIDDEN_LAYER_IDX + 1].float().mean(dim=1).detach()
            del out
        for n, mod in self.all_lora_mods: mod.weight.data.copy_(saved[n])
        del saved
        return logits, hs_ref

    # ---- Phase 1: GradProject -----------------------------------------------

    def unlearn_step(self, forget_question, forget_answer,
                     retain_pool, retain_start_idx, tokenizer, steps=15):
        self.model.train()
        device = get_input_device(self.model)
        forget_enc, forget_labels = self._encode(
            forget_question, forget_answer, tokenizer, device)
        opt = torch.optim.Adam(
            [m.weight for _, m in self.target_modules], lr=self.lr)

        for step in range(steps):
            rs = retain_pool[(retain_start_idx + step) % len(retain_pool)]
            retain_enc, _ = self._encode(rs['question'], rs['answer'], tokenizer, device)

            opt.zero_grad()
            self.model(**forget_enc, labels=forget_labels).loss.backward()
            f_grads = {n: m.weight.grad.clone()
                       for n, m in self.target_modules if m.weight.grad is not None}

            opt.zero_grad()
            ref_logits = self._get_finetuned_logits(retain_enc)
            ref_probs  = F.softmax(ref_logits / self.kl_temp, dim=-1)
            del ref_logits
            self.model.train()
            kl_loss = F.kl_div(
                F.log_softmax(self.model(**retain_enc).logits.float() / self.kl_temp, dim=-1),
                ref_probs, reduction='batchmean')
            del ref_probs
            kl_loss.backward()

            with torch.no_grad():
                for name, mod in self.target_modules:
                    g_r = mod.weight.grad
                    if g_r is None: continue
                    g_f = f_grads.get(name)
                    if g_f is None: mod.weight.grad.zero_(); continue
                    g_f = g_f.to(g_r.device)
                    dot = (g_f * g_r).sum()
                    g_f_proj = g_f - (dot / (g_r * g_r).sum().clamp(1e-12)) * g_r
                    mod.weight.grad.copy_(-g_f_proj)

            torch.nn.utils.clip_grad_norm_(
                [m.weight for _, m in self.target_modules], self.max_grad_norm)
            opt.step()
            del f_grads, retain_enc

        torch.cuda.empty_cache()

    # ---- Phase 2a: MLP SVD --------------------------------------------------

    def svd_prune(self, forget_dataset, tokenizer, n_score_samples=20):
        if not self.do_svd_prune: return
        print(f'\nPhase 2a -- MLP SVD ({self.svd_prune_ratio:.0%})...')
        device = get_input_device(self.model)
        self.model.eval()

        def bkey(n): return re.sub(r'\.lora_[AB].*', '', n)

        a_by, b_by = {}, {}
        for n, m in self.lora_a_modules:
            if any(t in n for t in self.SVD_MLP_TYPES): a_by[bkey(n)] = (n, m)
        for n, m in self.lora_b_modules:
            if any(t in n for t in self.SVD_MLP_TYPES): b_by[bkey(n)] = (n, m)
        keys = set(a_by) & set(b_by)
        print(f'  {len(keys)} MLP pairs')

        sig = {}
        for i in tqdm(range(min(n_score_samples, len(forget_dataset))),
                      desc='Scoring MLP dims'):
            s = forget_dataset[i]
            enc, lbl = self._encode(s['question'], s['answer'], tokenizer, device)
            self.model.zero_grad()
            self.model(**enc, labels=lbl).loss.backward()
            with torch.no_grad():
                for k in keys:
                    _, mb = b_by[k]
                    if mb.weight.grad is None: continue
                    sig[k] = sig.get(k, 0) + mb.weight.grad.float().norm(dim=0).cpu()
        self.model.zero_grad(); gc.collect(); torch.cuda.empty_cache()

        zeroed = 0
        for k in keys:
            _, ma = a_by[k]; _, mb = b_by[k]
            rank = ma.weight.shape[0]
            n_p  = max(1, int(self.svd_prune_ratio * rank))
            dims = sig.get(k, torch.ones(rank)).argsort(descending=True)[:n_p]
            with torch.no_grad():
                mb.weight.data[:, dims] = 0.0
                ma.weight.data[dims, :] = 0.0
            zeroed += n_p
        print(f'  Zeroed {zeroed} MLP rank dims.')

    # ---- Phase 2b: o_proj micro-prune ---------------------------------------

    def attn_micro_prune(self, forget_dataset, tokenizer, n_score_samples=20):
        if not self.attn_a_modules:
            print('  No o_proj lora modules -- skipping.'); return
        print(f'\nPhase 2b -- o_proj micro-prune ({self.attn_prune_ratio:.0%})...')
        device = get_input_device(self.model)
        self.model.eval()

        def bkey(n): return re.sub(r'\.lora_[AB].*', '', n)

        a_by, b_by = {}, {}
        for n, m in self.attn_a_modules: a_by[bkey(n)] = (n, m)
        for n, m in self.attn_b_modules: b_by[bkey(n)] = (n, m)
        keys = set(a_by) & set(b_by)

        sig = {}
        for i in tqdm(range(min(n_score_samples, len(forget_dataset))),
                      desc='Scoring o_proj dims'):
            s = forget_dataset[i]
            enc, lbl = self._encode(s['question'], s['answer'], tokenizer, device)
            self.model.zero_grad()
            self.model(**enc, labels=lbl).loss.backward()
            with torch.no_grad():
                for k in keys:
                    _, mb = b_by[k]
                    if mb.weight.grad is None: continue
                    sig[k] = sig.get(k, 0) + mb.weight.grad.float().norm(dim=0).cpu()
        self.model.zero_grad(); gc.collect(); torch.cuda.empty_cache()

        zeroed = 0
        for k in keys:
            _, ma = a_by[k]; _, mb = b_by[k]
            rank = ma.weight.shape[0]
            n_p  = max(1, int(self.attn_prune_ratio * rank))
            dims = sig.get(k, torch.ones(rank)).argsort(descending=True)[:n_p]
            with torch.no_grad():
                mb.weight.data[:, dims] = 0.0
                ma.weight.data[dims, :] = 0.0
            zeroed += n_p
        print(f'  Zeroed {zeroed} o_proj rank dims.')

    # ---- Phase 2.5A: compute forget task vectors ----------------------------

    def compute_forget_task_vector(self, forget_dataset, tokenizer, n_score_samples=20):
        print(f'\nPhase 2.5A -- Computing forget task vectors '
              f'({n_score_samples} scoring samples)...')
        device = get_input_device(self.model)
        self.model.eval()

        b_mods = {}
        for name, mod in self.model.named_modules():
            if not isinstance(mod, nn.Linear): continue
            m = re.search(r'layers\.(\d+)', name)
            if not m or not (self.mid_layer_start <= int(m.group(1)) <= self.mid_layer_end): continue
            if not any(mt in name for mt in self.REPAIR_MODULE_TYPES): continue
            if 'lora_B' in name: b_mods[name] = mod

        print(f'  {len(b_mods)} lora_B targets (all 7 module types, '
              f'layers {self.mid_layer_start}-{self.mid_layer_end})')

        saved_req = {n: mod.weight.requires_grad for n, mod in b_mods.items()}
        for mod in b_mods.values(): mod.weight.requires_grad_(True)

        sig = {}
        try:
            for i in tqdm(range(min(n_score_samples, len(forget_dataset))),
                          desc='Scoring forget dims (all lora_B)'):
                s = forget_dataset[i]
                enc, lbl = self._encode(s['question'], s['answer'], tokenizer, device)
                self.model.zero_grad()
                self.model(**enc, labels=lbl).loss.backward()
                with torch.no_grad():
                    for name, mod in b_mods.items():
                        if mod.weight.grad is None: continue
                        sig[name] = sig.get(name, 0) + mod.weight.grad.float().norm(dim=0).cpu()
        finally:
            for name, mod in b_mods.items(): mod.weight.requires_grad_(saved_req[name])
            self.model.zero_grad()
            gc.collect(); torch.cuda.empty_cache()

        self.forget_task_vec = {}
        total_masked = 0
        for name, mod in b_mods.items():
            scores = sig.get(name)
            if scores is None: continue
            rank     = mod.weight.shape[1]
            n_top    = max(1, rank // 2)
            top_dims = scores.argsort(descending=True)[:n_top]
            mask     = torch.zeros(rank, dtype=torch.float32)
            mask[top_dims] = 1.0
            ft_w = self.finetuned_weights.get(name)
            if ft_w is None: continue
            self.forget_task_vec[name] = ft_w * mask.unsqueeze(0)
            total_masked += n_top

        n_mods = len(self.forget_task_vec)
        print(f'  Task vectors built for {n_mods} modules  | '
              f'{total_masked} forget-coded dims (~{total_masked//max(1,n_mods)} per module)')

    # ---- Phase 2.5B: apply task vector negation -----------------------------

    def apply_task_vector_negation(self, alpha=1.0):
        if not self.forget_task_vec:
            print('  WARNING: forget_task_vec empty. Skipping.'); return
        print(f'\nPhase 2.5B -- Task vector negation (alpha={alpha})...')
        n_applied = 0
        with torch.no_grad():
            for name, mod in self.model.named_modules():
                if not isinstance(mod, nn.Linear): continue
                tv = self.forget_task_vec.get(name)
                if tv is None: continue
                mod.weight.data.sub_(alpha * tv.to(mod.weight.device))
                n_applied += 1
        print(f'  Subtracted alpha*forget_task_vec from {n_applied} lora_B modules.')

    # ---- Phase 3: retain repair ---------------------------------------------

    def retain_repair(
        self, retain_dataset, tokenizer,
        forget_dataset=None,
        n_steps=150, repair_lr=8e-5,
        kl_weight=0.60, hs_weight=0.25,
        forget_reg_weight=0.10, tv_weight=0.05,
        repair_kl_temp=2.0, final_lr=5e-6,
    ):
        # Loss = kl_weight*KL + hs_weight*HS + forget_reg_weight*(-H) + tv_weight*TV
        # BUG FIX 1: tv_sims now aggregated via torch.stack().mean()
        # BUG FIX 2: clip_grad_norm_ added before opt.step()
        print(f'\nPhase 3 -- Hybrid repair  kl={kl_weight} hs={hs_weight} '
              f'freg={forget_reg_weight} tv={tv_weight}  '
              f'temp={repair_kl_temp}  lr {repair_lr:.0e}->{final_lr:.0e}  steps={n_steps}')
        device = get_input_device(self.model)
        self.model.train()

        repair_mods = []
        for name, mod in self.model.named_modules():
            if not isinstance(mod, nn.Linear): continue
            m = re.search(r'layers\.(\d+)', name)
            if not m or not (7 <= int(m.group(1)) <= 20): continue
            if not any(mt in name for mt in self.REPAIR_MODULE_TYPES): continue
            if 'lora_A' in name or 'lora_B' in name:
                mod.weight.requires_grad = True
                repair_mods.append((name, mod))
        print(f'  Repair modules: {len(repair_mods)}')

        has_tv = bool(self.forget_task_vec) and tv_weight > 0
        print(f'  TV reg: {"active (" + str(len(self.forget_task_vec)) + " task vecs)" if has_tv else "inactive"}')

        heavy_set = set(self.UNLEARN_MODULE_TYPES)
        heavy_p = [m.weight for n, m in repair_mods if any(t in n for t in heavy_set)]
        light_p = [m.weight for n, m in repair_mods if not any(t in n for t in heavy_set)]
        opt = torch.optim.Adam([
            {'params': heavy_p, 'lr': repair_lr},
            {'params': light_p, 'lr': repair_lr * 0.3},
        ])

        samples = [retain_dataset[i] for i in range(len(retain_dataset))]
        random.shuffle(samples)
        f_samples = None
        if forget_dataset is not None and forget_reg_weight > 0:
            f_samples = [forget_dataset[i] for i in range(len(forget_dataset))]
            random.shuffle(f_samples)
            print(f'  Forget reg: {len(f_samples)} samples')

        for step in range(n_steps):
            cos = 0.5 * (1.0 + math.cos(math.pi * step / n_steps))
            opt.param_groups[0]['lr'] = final_lr + (repair_lr       - final_lr) * cos
            opt.param_groups[1]['lr'] = final_lr + (repair_lr * 0.3 - final_lr) * cos

            sample = samples[step % len(samples)]
            enc, _ = self._encode(sample['question'], sample['answer'], tokenizer, device)
            opt.zero_grad()

            ref_logits, ref_hs = self._get_finetuned_outputs(enc)
            ref_probs = F.softmax(ref_logits / repair_kl_temp, dim=-1)
            del ref_logits

            self.model.train()
            out          = self.model(**enc, output_hidden_states=True)
            model_logits = out.logits.float()
            curr_hs      = out.hidden_states[self.HIDDEN_LAYER_IDX + 1].float().mean(dim=1)
            del out

            kl_loss = F.kl_div(
                F.log_softmax(model_logits / repair_kl_temp, dim=-1),
                ref_probs, reduction='batchmean')

            hs_loss = 1.0 - F.cosine_similarity(curr_hs, ref_hs, dim=-1).mean()

            forget_reg_loss = torch.tensor(0.0, device=device)
            if f_samples is not None:
                fs = f_samples[step % len(f_samples)]
                f_enc, f_labels = self._encode(fs['question'], fs['answer'], tokenizer, device)
                f_logits = self.model(**f_enc).logits.float()
                f_mask   = (f_labels[0] != -100)
                if f_mask.any():
                    f_ans = f_logits[0][f_mask]
                    f_lp  = F.log_softmax(f_ans, dim=-1)
                    f_ent = -(f_lp.exp() * f_lp).sum(dim=-1).mean()
                    forget_reg_loss = -f_ent
                del f_logits

            # BUG FIX: tv_sims aggregated into tv_reg_loss
            tv_reg_loss = torch.tensor(0.0, device=device)
            if has_tv:
                tv_sims = []
                for name, mod in self.model.named_modules():
                    if not isinstance(mod, nn.Linear): continue
                    tv = self.forget_task_vec.get(name)
                    if tv is None: continue
                    tv_dev    = tv.to(mod.weight.device)
                    curr_flat = mod.weight.float().view(1, -1)
                    tv_flat   = tv_dev.view(1, -1)
                    sim = F.cosine_similarity(curr_flat, tv_flat, dim=1)
                    tv_sims.append(sim.clamp(min=0.0).to(device))
                if tv_sims:
                    tv_reg_loss = torch.stack(tv_sims).mean()  # BUG FIX

            total = (kl_weight         * kl_loss
                   + hs_weight         * hs_loss
                   + forget_reg_weight * forget_reg_loss
                   + tv_weight         * tv_reg_loss)
            del ref_probs, model_logits, curr_hs, ref_hs

            total.backward()
            torch.nn.utils.clip_grad_norm_(  # BUG FIX
                [m.weight for _, m in repair_mods], self.max_grad_norm)
            opt.step()

            if (step + 1) % 25 == 0:
                torch.cuda.empty_cache()
                freg_v = forget_reg_loss.item() if isinstance(forget_reg_loss, torch.Tensor) else 0.0
                tv_v   = tv_reg_loss.item()   if isinstance(tv_reg_loss,   torch.Tensor) else 0.0
                print(f'  step {step+1:3d}/{n_steps}  '
                      f'kl={kl_loss.item():.4f}  hs={hs_loss.item():.4f}  '
                      f'freg={freg_v:.4f}  tv={tv_v:.4f}  '
                      f'total={total.item():.4f}  '
                      f'lr={opt.param_groups[0]["lr"]:.2e}')

        print('Hybrid repair complete.')


print('RecursiveMAAT_v9 (v9.3 TOFU), helpers -- loaded.')


## 5 . Run MA'AT v9.3 Unlearning (TOFU forget05 / retain95)

**Scaling vs factify-500:**

| Param | factify 500+500 | TOFU 200+3800 | Reason |
|---|---|---|---|
| `UNLEARN_STEPS` | 12 | **15** | 200x15=3000 updates; slightly more steps to compensate for smaller forget set |
| `SVD_SCORE_SAMPLES` | 60 | **40** | 20% of 200 forget samples |
| `TV_SCORE_SAMPLES` | 60 | **40** | Same |
| `REPAIR_STEPS` | 300 | **400** | 3800 retain samples need more cycling |

In [ ]:
import os, shutil, gc, random, ast
import torch
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_from_disk

MODEL_NAME        = 'meta-llama/Llama-3.2-3B'
LORA_ADAPTER_PATH = '/kaggle/working/lora_adapter'
UNLEARNED_ADAPTER = '/kaggle/working/lora_adapter_unlearned'
DATA_DIR          = '/kaggle/working/data_splits'

# -- Hyperparameters ----------------------------------------------------------
MID_LAYER_START  = 7
MID_LAYER_END    = 20
LEARNING_RATE    = 1e-5
MAX_GRAD_NORM    = 1.0
UNLEARN_STEPS    = 15    # 200x15=3000 primary gradient updates
KL_TEMP          = 0.7

# Rephrase augmentation (TOFU has no rephrases field -- silently skipped)
USE_REPHRASES  = True
REPHRASE_STEPS = 8
MAX_REPHRASES  = 2

# Phase 2a -- MLP SVD
DO_SVD_PRUNE      = True
SVD_PRUNE_RATIO   = 0.15
SVD_SCORE_SAMPLES = 40   # 20% of 200 forget samples

# Phase 2b -- disabled (destructive at rank=32)
DO_ATTN_PRUNE = False

# Phase 2.5A/B -- Task vector negation
DO_TASK_VEC      = True
TV_SCORE_SAMPLES = 40
TV_ALPHA         = 1.0

# Phase 3 -- Forget-aware hybrid repair with TV reg
DO_RETAIN_REPAIR  = True
REPAIR_STEPS      = 400  # increased: 3800 retain samples need more cycling
REPAIR_LR         = 8e-5
KL_WEIGHT         = 0.60
HS_WEIGHT         = 0.25
FORGET_REG_WEIGHT = 0.10
TV_WEIGHT         = 0.05
REPAIR_KL_TEMP    = 2.0
FINAL_LR          = 5e-6
# -----------------------------------------------------------------------------

gc.collect(); torch.cuda.empty_cache()
random.seed(42)

forget_dataset = load_from_disk(f'{DATA_DIR}/forget_dataset')
retain_dataset = load_from_disk(f'{DATA_DIR}/retain_dataset')
retain_list    = [retain_dataset[i] for i in range(len(retain_dataset))]
random.shuffle(retain_list)
print(f'Forget: {len(forget_dataset)} | Retain: {len(retain_dataset)}')
assert len(forget_dataset) == 200,  (
    f'Expected 200 forget samples (TOFU forget05), got {len(forget_dataset)}')
assert len(retain_dataset) == 3800, (
    f'Expected 3800 retain samples (TOFU retain95), got {len(retain_dataset)}')

tokenizer = AutoTokenizer.from_pretrained(LORA_ADAPTER_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Loading base model + LoRA adapter...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map='auto'
)
model = PeftModel.from_pretrained(
    base_model, LORA_ADAPTER_PATH, is_trainable=True
)
model.eval()

maat = RecursiveMAAT_v9(
    model,
    mid_layer_start        = MID_LAYER_START,
    mid_layer_end          = MID_LAYER_END,
    learning_rate          = LEARNING_RATE,
    max_grad_norm          = MAX_GRAD_NORM,
    kl_temp                = KL_TEMP,
    do_svd_prune           = DO_SVD_PRUNE,
    svd_prune_ratio        = SVD_PRUNE_RATIO,
    attn_prune_ratio       = 0.02,
    attn_prune_layer_start = 14,
)

def get_rephrases(sample):
    raw = sample.get('rephrases', [])
    if isinstance(raw, list):
        return [r for r in raw if r and str(r).strip()]
    if isinstance(raw, str):
        try:
            return [r for r in ast.literal_eval(raw) if r and str(r).strip()]
        except Exception:
            return [raw.strip()] if raw.strip() else []
    return []

# -- Phase 1: GradProject + rephrase augmentation ----------------------------
n_reph = 0
print(f'\n[Phase 1] {len(forget_dataset)} samples x {UNLEARN_STEPS} steps')
print(f'  Primary updates: {len(forget_dataset) * UNLEARN_STEPS:,} '
      f'| Max rephrase: {len(forget_dataset) * MAX_REPHRASES * REPHRASE_STEPS:,}')
for i, fs in enumerate(forget_dataset):
    maat.unlearn_step(
        forget_question  = fs['question'],
        forget_answer    = fs['answer'],
        retain_pool      = retain_list,
        retain_start_idx = i * (MAX_REPHRASES + 1),
        tokenizer        = tokenizer,
        steps            = UNLEARN_STEPS,
    )
    if USE_REPHRASES:
        for j, rq in enumerate(get_rephrases(fs)[:MAX_REPHRASES]):
            maat.unlearn_step(
                forget_question  = rq,
                forget_answer    = fs['answer'],
                retain_pool      = retain_list,
                retain_start_idx = i * (MAX_REPHRASES + 1) + j + 1,
                tokenizer        = tokenizer,
                steps            = REPHRASE_STEPS,
            )
            n_reph += 1
    if (i + 1) % 50 == 0:
        torch.cuda.empty_cache()
        print(f'  [{i+1}/{len(forget_dataset)}]  rephrases so far: {n_reph}')
print(f'Phase 1 complete. Rephrase variants: {n_reph}\n')

# -- Phase 2a -----------------------------------------------------------------
if DO_SVD_PRUNE:
    maat.svd_prune(forget_dataset, tokenizer, n_score_samples=SVD_SCORE_SAMPLES)

# -- Phase 2b -----------------------------------------------------------------
if DO_ATTN_PRUNE:
    maat.attn_micro_prune(forget_dataset, tokenizer, n_score_samples=20)

# -- Phase 2.5A ---------------------------------------------------------------
if DO_TASK_VEC:
    maat.compute_forget_task_vector(forget_dataset, tokenizer,
                                    n_score_samples=TV_SCORE_SAMPLES)

# -- Phase 2.5B ---------------------------------------------------------------
if DO_TASK_VEC:
    maat.apply_task_vector_negation(alpha=TV_ALPHA)

# -- Spot-check ---------------------------------------------------------------
_eos = list({tokenizer.eos_token_id,
             tokenizer.convert_tokens_to_ids('<|eot_id|>')})
_eos = [e for e in _eos if e is not None and e >= 0]
model.eval()
_dev = get_input_device(model)

print('\n[Spot-check] Coherence after Phase 2 + 2.5:')
any_broken = 0
severe_broken = 0
for _i in range(4):
    _s = retain_list[_i]
    _e = tokenizer(format_prompt_eval(_s['question']),
                   return_tensors='pt', truncation=True, max_length=256)
    _e = {k: v.to(_dev) for k, v in _e.items()}
    with torch.no_grad():
        _g = model.generate(**_e, max_new_tokens=50, do_sample=False,
                             eos_token_id=_eos,
                             pad_token_id=tokenizer.eos_token_id,
                             repetition_penalty=1.3)
    _r = tokenizer.decode(_g[0][_e['input_ids'].shape[1]:],
                          skip_special_tokens=True).strip()
    _mod  = is_output_broken(_r, severe_only=False)
    _sev  = is_output_broken(_r, severe_only=True)
    if _mod: any_broken    += 1
    if _sev: severe_broken += 1
    mark = 'SEVERE' if _sev else ('MODERATE' if _mod else 'OK')
    print(f'  [{mark}] Q: {_s["question"][:80]}')
    print(f'         A: {_r[:120]}\n')

if severe_broken >= 3:
    print('AUTO-ABORT: >=3/4 samples severely broken. '
          'Try reducing TV_ALPHA or SVD_PRUNE_RATIO.')
elif any_broken >= 2:
    print(f'WARNING: {any_broken}/4 samples show moderate damage. '
          'Phase 3 repair will recover this.')
else:
    print('Spot-check passed cleanly.')

if severe_broken < 3:
    if DO_RETAIN_REPAIR:
        maat.retain_repair(
            retain_dataset, tokenizer,
            forget_dataset     = forget_dataset,
            n_steps            = REPAIR_STEPS,
            repair_lr          = REPAIR_LR,
            kl_weight          = KL_WEIGHT,
            hs_weight          = HS_WEIGHT,
            forget_reg_weight  = FORGET_REG_WEIGHT,
            tv_weight          = TV_WEIGHT,
            repair_kl_temp     = REPAIR_KL_TEMP,
            final_lr           = FINAL_LR,
        )

    if os.path.exists(UNLEARNED_ADAPTER): shutil.rmtree(UNLEARNED_ADAPTER)
    model.save_pretrained(UNLEARNED_ADAPTER)
    tokenizer.save_pretrained(UNLEARNED_ADAPTER)
    print(f'\nUnlearned adapter -> {UNLEARNED_ADAPTER}')


## 6 . Merge Unlearned Adapter into Full Model

In [ ]:
import gc, shutil, os
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME        = 'meta-llama/Llama-3.2-3B'
UNLEARNED_ADAPTER = '/kaggle/working/lora_adapter_unlearned'
MERGED_MODEL_PATH = '/kaggle/working/Llama-3.2-3B-TOFU-Unlearned-v9'

try:
    del model, maat, base_model
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()

print('Merging unlearned adapter -> full model...')
tokenizer = AutoTokenizer.from_pretrained(UNLEARNED_ADAPTER)
base_for_merge = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map='auto'
)
merged = PeftModel.from_pretrained(
    base_for_merge, UNLEARNED_ADAPTER).merge_and_unload()

if os.path.exists(MERGED_MODEL_PATH): shutil.rmtree(MERGED_MODEL_PATH)
merged.save_pretrained(MERGED_MODEL_PATH)
tokenizer.save_pretrained(MERGED_MODEL_PATH)
print(f'Merged model -> {MERGED_MODEL_PATH}')

del merged, base_for_merge
gc.collect(); torch.cuda.empty_cache()


## 7 . Generate & Save Answers (for LLM-as-Judge)

TOFU has no label column -- `label` field removed from records vs original factify version.

In [ ]:
import json, os, gc
import torch
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM

MERGED_MODEL_PATH = '/kaggle/working/Llama-3.2-3B-TOFU-Unlearned-v9'
DATA_DIR          = '/kaggle/working/data_splits'
OUT_DIR           = '/kaggle/working/eval_inputs'
MAX_NEW_TOKENS    = 100

os.makedirs(OUT_DIR, exist_ok=True)
gc.collect(); torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MERGED_MODEL_PATH, torch_dtype=torch.float16, device_map='auto'
)
model.eval()
input_device = get_input_device(model)
torch.manual_seed(42)

eos_ids = list({tokenizer.eos_token_id,
                tokenizer.convert_tokens_to_ids('<|eot_id|>')})
eos_ids = [e for e in eos_ids if e is not None and e >= 0]


def generate_answer(question):
    prompt = format_prompt_eval(question)
    inputs = tokenizer(prompt, return_tensors='pt',
                       truncation=True, max_length=512).to(input_device)
    with torch.no_grad():
        gen = model.generate(
            **inputs,
            max_new_tokens     = MAX_NEW_TOKENS,
            do_sample          = False,
            repetition_penalty = 1.3,
            eos_token_id       = eos_ids,
            pad_token_id       = tokenizer.eos_token_id,
        )
    new_ids = gen[0][inputs['input_ids'].shape[1]:]
    return clean_output(tokenizer.decode(new_ids, skip_special_tokens=True).strip())


def collect_answers(dataset, split_name):
    records = []
    print(f'[{split_name}] {len(dataset)} samples...')
    for i, sample in enumerate(dataset):
        records.append({
            'split'        : split_name,
            'idx'          : i,
            'question'     : sample['question'],
            'ground_truth' : sample['answer'],
            'model_answer' : generate_answer(sample['question']),
        })
        if (i + 1) % 10 == 0:
            torch.cuda.empty_cache()
            print(f'  [{i+1}/{len(dataset)}]')
    return records


forget_dataset = load_from_disk(f'{DATA_DIR}/forget_dataset')
retain_dataset = load_from_disk(f'{DATA_DIR}/retain_dataset')

forget_records = collect_answers(forget_dataset, 'forget')
retain_records = collect_answers(retain_dataset, 'retain')
all_records    = forget_records + retain_records

for fname, records in [
    ('forget_answers.json', forget_records),
    ('retain_answers.json', retain_records),
    ('all_answers.json',    all_records),
]:
    with open(os.path.join(OUT_DIR, fname), 'w') as f:
        json.dump(records, f, indent=2, ensure_ascii=False)

print(f'\nSaved to {OUT_DIR}/')
for fname in ['forget_answers.json', 'retain_answers.json', 'all_answers.json']:
    p = os.path.join(OUT_DIR, fname)
    print(f'  {p}  ({os.path.getsize(p)/1024:.1f} KB)')
print('\n-- Sample forget --')
print(json.dumps(forget_records[0], indent=2))
print('\n-- Sample retain --')
print(json.dumps(retain_records[0], indent=2))


## 8 . ROUGE Evaluation

| Metric | Forget set | Retain set |
|---|---|---|
| ROUGE down vs fine-tuned baseline | Good -- forget content removed | Bad -- retain knowledge lost |
| ROUGE approx fine-tuned baseline | Bad -- content not forgotten | Good -- retain preserved |

In [ ]:
import evaluate, json, os, gc
import torch
from tqdm import tqdm
from datasets import load_from_disk

MERGED_MODEL_PATH = '/kaggle/working/Llama-3.2-3B-TOFU-Unlearned-v9'
DATA_DIR          = '/kaggle/working/data_splits'
OUT_DIR           = '/kaggle/working/eval_inputs'
ROUGE_OUT         = f'{OUT_DIR}/rouge_scores.json'

rouge_metric = evaluate.load('rouge')

try:
    _ = model
    print('Using model already in memory.')
except NameError:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_PATH)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        MERGED_MODEL_PATH, torch_dtype=torch.float16, device_map='auto')
    model.eval()
    print('Model loaded from disk.')

eos_ids = list({tokenizer.eos_token_id,
                tokenizer.convert_tokens_to_ids('<|eot_id|>')})
eos_ids = [e for e in eos_ids if e is not None and e >= 0]
input_device = next(model.parameters()).device


def _generate(question):
    import re
    prompt = format_prompt_eval(question)
    inputs = tokenizer(prompt, return_tensors='pt',
                       truncation=True, max_length=512).to(input_device)
    with torch.no_grad():
        gen = model.generate(
            **inputs, max_new_tokens=100, do_sample=False,
            repetition_penalty=1.3, eos_token_id=eos_ids,
            pad_token_id=tokenizer.eos_token_id,
        )
    raw = tokenizer.decode(gen[0][inputs['input_ids'].shape[1]:],
                           skip_special_tokens=True).strip()
    return re.sub(r'\s*(CLIIIK|\?>|\];|\]|/\*|<!|\{\{|\}\}).*$',
                  '', raw, flags=re.DOTALL).strip()


def compute_rouge(dataset_path, split_name):
    print(f'\n-- {split_name} --')
    ds = load_from_disk(dataset_path)
    preds, refs = [], []
    for sample in tqdm(ds, desc=f'{split_name} generation'):
        preds.append(_generate(sample['question']))
        refs.append(sample['answer'])
    scores = rouge_metric.compute(predictions=preds, references=refs)
    print(f'  ROUGE-1 : {scores["rouge1"]:.4f}')
    print(f'  ROUGE-2 : {scores["rouge2"]:.4f}')
    print(f'  ROUGE-L : {scores["rougeL"]:.4f}')
    return {'split': split_name, **scores}

forget_scores = compute_rouge(f'{DATA_DIR}/forget_dataset', 'Forget (unlearned)')
retain_scores = compute_rouge(f'{DATA_DIR}/retain_dataset', 'Retain (preserved)')

summary = {'forget': forget_scores, 'retain': retain_scores}
os.makedirs(OUT_DIR, exist_ok=True)
with open(ROUGE_OUT, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'\nROUGE scores saved -> {ROUGE_OUT}')

print('\n====== ROUGE Summary ============================')
print(f'{"Metric":<12}  {"Forget":>10}  {"Retain":>10}')
print('-' * 36)
for m in ['rouge1', 'rouge2', 'rougeL']:
    print(f'{m:<12}  {forget_scores[m]:>10.4f}  {retain_scores[m]:>10.4f}')

print('\n====== BERTScore Comparison =====================')
print('PRE-unlearning (forget05 baseline):')
try:
    with open(f'{OUT_DIR}/bertscore_pre_unlearning.json') as fp:
        pre = json.load(fp)['summary']
    print(f'  P={pre["precision"]:.4f}  R={pre["recall"]:.4f}  F1={pre["f1"]:.4f}')
    print('POST-unlearning: run Section 9 below for full BERTScore comparison.')
except FileNotFoundError:
    print('  (pre-unlearning file not found; rerun Section 3)')


## 9 . BERTScore POST-unlearning + Delta Report

Runs BERTScore on both `forget05` and `retain95` after unlearning and prints the
delta vs the pre-unlearning baseline captured in Section 3.

**Targets:**
- `forget05` BERTScore F1 should **drop** significantly (content removed)
- `retain95` BERTScore F1 should remain **high** (knowledge preserved)

In [ ]:
import json, gc, os
import torch
from datasets import load_from_disk
from bert_score import score as bert_score_fn

MERGED_MODEL_PATH = '/kaggle/working/Llama-3.2-3B-TOFU-Unlearned-v9'
DATA_DIR          = '/kaggle/working/data_splits'
OUT_DIR           = '/kaggle/working/eval_inputs'

try:
    _ = model
    print('Using merged model already in memory.')
except NameError:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_PATH)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        MERGED_MODEL_PATH, torch_dtype=torch.float16, device_map='auto')
    model.eval()

post_device = next(model.parameters()).device
eos_ids_post = list({tokenizer.eos_token_id,
                     tokenizer.convert_tokens_to_ids('<|eot_id|>')})
eos_ids_post = [e for e in eos_ids_post if e is not None and e >= 0]


def _post_gen(question):
    import re
    prompt = format_prompt_eval(question)
    inp = tokenizer(prompt, return_tensors='pt',
                    truncation=True, max_length=512).to(post_device)
    with torch.no_grad():
        gen = model.generate(
            **inp, max_new_tokens=100, do_sample=False,
            repetition_penalty=1.3, eos_token_id=eos_ids_post,
            pad_token_id=tokenizer.eos_token_id,
        )
    raw = tokenizer.decode(gen[0][inp['input_ids'].shape[1]:],
                           skip_special_tokens=True).strip()
    return re.sub(r'\s*(CLIIIK|\?>|\];|\]|/\*|<!|\{\{|\}\}).*$',
                  '', raw, flags=re.DOTALL).strip()


def run_bertscore_split(dataset_path, split_name, max_samples=None):
    ds = load_from_disk(dataset_path)
    if max_samples is not None:
        ds = ds.select(range(min(max_samples, len(ds))))
    preds, refs = [], []
    print(f'Generating {len(ds)} answers for {split_name}...')
    for i, sample in enumerate(ds):
        preds.append(_post_gen(sample['question']))
        refs.append(sample['answer'])
        if (i + 1) % 20 == 0:
            torch.cuda.empty_cache()
            print(f'  [{i+1}/{len(ds)}]')
    print(f'Computing BERTScore for {split_name}...')
    P, R, F1 = bert_score_fn(preds, refs, lang='en',
                              model_type='roberta-large',
                              device=str(post_device), verbose=False)
    result = {
        'stage': 'post_unlearning', 'split': split_name,
        'n_samples': len(ds),
        'precision': round(P.mean().item(), 4),
        'recall'   : round(R.mean().item(), 4),
        'f1'       : round(F1.mean().item(), 4),
    }
    return result


# Run on forget05 (all 200) and a 200-sample subset of retain95 for speed
post_forget = run_bertscore_split(f'{DATA_DIR}/forget_dataset', 'forget05')
post_retain = run_bertscore_split(f'{DATA_DIR}/retain_dataset', 'retain95', max_samples=200)

# Load pre-unlearning baseline
try:
    with open(f'{OUT_DIR}/bertscore_pre_unlearning.json') as fp:
        pre_summary = json.load(fp)['summary']
    pre_forget_f1 = pre_summary['f1']
except FileNotFoundError:
    pre_forget_f1 = None
    print('WARNING: pre-unlearning BERTScore not found; rerun Section 3.')

# Save post-unlearning results
with open(f'{OUT_DIR}/bertscore_post_unlearning.json', 'w') as fp:
    json.dump({'forget05': post_forget, 'retain95': post_retain}, fp, indent=2)

print('\n====== BERTScore Report ================================')
print(f'{"Split":<12} {"Stage":<18} {"P":>8} {"R":>8} {"F1":>8}')
print('-' * 56)
if pre_forget_f1 is not None:
    pre = pre_summary
    print(f'{"forget05":<12} {"pre-unlearning":<18} '
          f'{pre["precision"]:>8.4f} {pre["recall"]:>8.4f} {pre["f1"]:>8.4f}')
print(f'{"forget05":<12} {"post-unlearning":<18} '
      f'{post_forget["precision"]:>8.4f} {post_forget["recall"]:>8.4f} {post_forget["f1"]:>8.4f}')
print(f'{"retain95":<12} {"post-unlearning":<18} '
      f'{post_retain["precision"]:>8.4f} {post_retain["recall"]:>8.4f} {post_retain["f1"]:>8.4f}')

if pre_forget_f1 is not None:
    delta = post_forget["f1"] - pre_forget_f1
    direction = "drop (good -- forgot)" if delta < -0.02 else \
                ("minimal change" if abs(delta) <= 0.02 else "increase (bad)")
    print(f'\nforget05 F1 delta: {delta:+.4f}  ({direction})')
    print(f'retain95 post F1: {post_retain["f1"]:.4f}  '
          f'(target: close to pre-unlearning value)')

print(f'\nResults saved -> {OUT_DIR}/bertscore_post_unlearning.json')
